In [17]:
import pandas as pd
import numpy as np
import datetime

In [18]:
df = pd.read_csv(
    '../data/wustl_iiot_2021.csv',
    parse_dates=['StartTime', 'LastTime']
)

In [19]:
negative_duration_mask = df['StartTime'] > df['LastTime']
df['StartTime'], df['LastTime'] = (
    np.where(negative_duration_mask, df['LastTime'], df['StartTime']),
    np.where(negative_duration_mask, df['StartTime'], df['LastTime']),
)

In [20]:
df['duration'] = df['LastTime'] - df['StartTime']
df['duration_s'] = df['duration'].dt.total_seconds().astype(int)


In [21]:
df[df['duration_s'] < 0]

,StartTime,LastTime,SrcAddr,DstAddr,Mean,Sport,Dport,SrcPkts,DstPkts,TotPkts,...,TotAppByte,SynAck,RunTime,sTos,SrcJitAct,DstJitAct,Traffic,Target,duration,duration_s


In [22]:
df['duration_s'].unique()

array([    0,     1,     4,     5,     3,     2,    60,     6, 16090,
        2984])

In [23]:
cols_to_avg = ['SrcPkts', 'DstPkts', 'TotPkts']   # columns to divide

df['_n'] = df['duration_s'] + 1         # rows per original

df_exp = df.loc[df.index.repeat(df['_n'])].copy()
df_exp['_offset'] = df_exp.groupby(level=0).cumcount()

df_exp['StartTime'] = df_exp['StartTime'] + pd.to_timedelta(df_exp['_offset'], unit='s')

# Divide chosen columns by n (row-wise broadcast)
df_exp[cols_to_avg] = df_exp[cols_to_avg].div(df_exp['_n'], axis=0)

df_exp = df_exp.drop(columns=['_n', '_offset']).reset_index(drop=True)

In [24]:
df_exp.drop(columns=['duration', 'duration_s', 'LastTime'], inplace=True)

In [25]:
cols = ['SrcPkts', 'DstPkts', 'TotPkts']

per_sec = (
    df.set_index('StartTime')
      .groupby(['SrcAddr', 'DstAddr'])
      .resample('1s', origin='start')[cols]
      .sum()
      .reset_index()
)

result = (
    per_sec.set_index('StartTime')
      .groupby(['SrcAddr', 'DstAddr'])
      .resample('60s', origin='start')[cols]
      .agg(list)
      .reset_index()
)

In [26]:
def pad60(v, k=60):
    a = np.zeros(k, dtype=float)
    n = min(len(v), k)
    a[:n] = v[:n]
    return a

for c in cols_to_avg:
    result[c] = result[c].apply(pad60)

In [27]:
result['bin_idx'] = result.groupby(['SrcAddr', 'DstAddr']).cumcount()

In [28]:
result

,SrcAddr,DstAddr,StartTime,SrcPkts,DstPkts,TotPkts,bin_idx
0,0,0,2019-08-19 09:46:08,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0
1,0,0,2019-08-19 09:47:08,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1
2,0,0,2019-08-19 09:48:08,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",2
3,0,0,2019-08-19 09:49:08,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",3
4,0,0,2019-08-19 09:50:08,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",4
...,...,...,...,...,...,...,...
19863,fe80::e9ed:931f:c2e0:1333,ff02::1:3,2019-08-19 16:35:15,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",401
19864,fe80::e9ed:931f:c2e0:1333,ff02::1:3,2019-08-19 16:36:15,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",402
19865,fe80::e9ed:931f:c2e0:1333,ff02::1:3,2019-08-19 16:37:15,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",403
19866,fe80::e9ed:931f:c2e0:1333,ff02::1:3,2019-08-19 16:38:15,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",404


In [29]:
result.to_csv('../data/wustl_iiot_2021_cleaned.csv', index=False)